# 模型比較實驗（Colab 版）

同一份 17 題 RAG 考卷，比較 Claude、Gemini 與地端開源模型（Ollama）的正確率、速度與費用。

**開始前兩個設定：**
1. 上方選單「執行階段」→「變更執行階段類型」→ 選 **T4 GPU** → 儲存（地端模型需要 GPU）
2. 左側欄的鑰匙圖示（Secrets）→ 新增兩個密鑰：`ANTHROPIC_API_KEY`、`GEMINI_API_KEY`，並打開「Notebook 存取權」

之後由上而下，一格一格按執行（▶）即可。

## 1. 下載程式

In [ ]:
!git clone -q https://github.com/chunyang130/model-benchmark.git
%cd model-benchmark
!ls

## 2. 讀取 API 金鑰（從 Colab Secrets，不會出現在畫面或程式裡）

In [ ]:
import os
from google.colab import userdata
for k in ['ANTHROPIC_API_KEY', 'GEMINI_API_KEY']:
    try:
        os.environ[k] = userdata.get(k)
        print(k, '已載入')
    except Exception as e:
        print(k, '沒有載入：', e)

## 3. 安裝 Ollama 並下載開源模型（約 3–5 分鐘）

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull qwen2.5:3b
!ollama pull qwen2.5:7b
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 4. 確認 Gemini 可用的模型名稱

如果下一步出現 404（找不到模型），從這裡的清單挑名字，改掉第 6 步的模型名稱。

In [ ]:
import json, urllib.request
req = urllib.request.Request('https://generativelanguage.googleapis.com/v1beta/models?pageSize=100', headers={'x-goog-api-key': os.environ.get('GEMINI_API_KEY', '')})
names = [m['name'].split('/')[-1] for m in json.load(urllib.request.urlopen(req))['models'] if 'generateContent' in m.get('supportedGenerationMethods', [])]
print('\n'.join(n for n in names if 'flash' in n))

## 5. Claude

In [ ]:
!python bench.py --provider claude --model claude-haiku-5-5
!python bench.py --provider claude --model claude-sonnet-5-5

## 6. Gemini（免費額度有每分鐘次數限制，程式遇到限流會自動等待重試）

In [ ]:
!python bench.py --provider gemini --model gemini-2.5-flash-lite
!python bench.py --provider gemini --model gemini-3.8-flash

## 7. 地端開源模型（Ollama，在這台 Colab GPU 上執行）

In [ ]:
!python bench.py --provider ollama --model qwen2.5:3b
!python bench.py --provider ollama --model qwen2.5:7b

## 8. 比較表與下載結果

執行後會下載 `results.zip`，請把它傳回給 Claude。

In [ ]:
!python bench.py --report
!zip -qr results.zip results
from google.colab import files
files.download('results.zip')